In [1]:
import pandas as pd 
import numpy as np 
import json 
import holidays

In [3]:
# Cargar datos
# Retiros por región
data_reg_dir = "../data/interim/demanda_regional_horaria.parquet"
df_reg = pd.read_parquet(
    data_reg_dir, 
    engine="pyarrow"
)
# BRE
data_bre_dir = "../data/raw/wp2_elec_input_sector_shares_raw.csv"
df_bre = pd.read_csv(data_bre_dir)
df_bre_2021 = df_bre[df_bre["año"] == 2021].copy()
# Pivotear
df_sec = df_bre_2021.pivot_table(
    index="región", columns="sector", values="valor", aggfunc="sum"
).reset_index()
sectores = ["Industrial", "Residencial", "Comercial", "Público", "Transporte"]
df_sec["total_BRE_TWh"] = df_sec[sectores].sum(axis=1) / 1e3


In [4]:
# Extraer datos del CEN del año 2021
df_reg["fecha_hora"] = pd.to_datetime(df_reg["fecha_hora"])
df_reg_2021 = df_reg[df_reg["fecha_hora"].dt.year == 2021].copy()

df_reg_anual = df_reg_2021.groupby("region")["demanda_mwh"].sum().reset_index()

df_reg_anual = df_reg_anual.rename(columns={
    "region": "región", 
    "demanda_mwh": "total_CEN_MWh"
})

df_reg_anual["total_CEN_TWh"] = df_reg_anual["total_CEN_MWh"] / 1e6

In [ ]:
# BRE y homologación de regiones con el CEN 
# Define la ruta a tu archivo JSON (ajústala según tu estructura de carpetas)
ruta_json_alias = "../data/raw/reg_alias.json" 

# Cargar el diccionario
with open(ruta_json_alias, 'r', encoding='utf-8') as f:
    dict_alias = json.load(f)

# Reemplazar los nombres en el DataFrame del CEN
# Los nombres que coincidan con las llaves del dict se cambiarán por el valor (nombre BNE)
df_reg_anual["región"] = df_reg_anual["región"].replace(dict_alias)

In [6]:
# Cruce de datos y cálculo de errores
df_comparacion = pd.merge(
    df_sec[["región", "total_BRE_TWh"]], 
    df_reg_anual[["región", "total_CEN_TWh"]], 
    on="región", 
    how="outer" 
)

df_comparacion["error_relativo_%"] = ((df_comparacion["total_CEN_TWh"] - df_comparacion["total_BRE_TWh"]) / df_comparacion["total_BRE_TWh"]) * 100

df_comparacion = df_comparacion.sort_values("error_relativo_%", ascending=False).reset_index(drop=True)

print("Comparación de Demanda Regional 2021 (CEN vs BRE):")
print(df_comparacion)

Comparación de Demanda Regional 2021 (CEN vs BRE):
   región  total_BRE_TWh  total_CEN_TWh  error_relativo_%
0      VS       4.960267       6.550070         32.050752
1      AP       0.333499       0.422618         26.722493
2      AT       3.237471       3.912984         20.865450
3      ML       2.831157       3.172226         12.046973
4      TA       2.213138       2.435457         10.045423
5      LI       4.765781       4.833909          1.429539
6      AN      17.456431      17.455934         -0.002848
7      BI       5.550964       5.177470         -6.728446
8      RM      23.585962      21.816263         -7.503186
9      NB       1.166782       1.030434        -11.685804
10     AR       2.067239       1.705542        -17.496637
11     LL       2.924532       2.264727        -22.561034
12     LR       1.512350       0.987941        -34.675087
13     CO       3.186498       2.034728        -36.145301
14     AI       0.204884            NaN               NaN
15     MA       0.337

In [ ]:
# Calcular mu y sigma
# Asegurarnos de tener el año como columna
df_reg["fecha_hora"] = pd.to_datetime(df_reg["fecha_hora"])
df_reg["año"] = df_reg["fecha_hora"].dt.year

# Calcular media y desviación estándar de la demanda horaria
df_params_cen = df_reg.groupby(["region", "año"]).agg(
    mu_total_MWh=("demanda_mwh", "mean"),
    sigma_total_MWh=("demanda_mwh", "std")
).reset_index()